In [20]:
import torch

example = torch.tensor([1.0, 2.0, 3.0])
print(example)
print(type(example))

tensor([1., 2., 3.])
<class 'torch.Tensor'>


In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("titanic.csv")
df_clean = df.drop_duplicates()
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].mean())
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
df_clean["has_cabin"] = df_clean["Cabin"].notnull().astype(int)
df_clean = df_clean.drop(columns=["Cabin"])

df_encoded = pd.get_dummies(df_clean, columns=["Sex", "Embarked"])

feature_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare", "has_cabin",
                 "Sex_female", "Sex_male", "Embarked_C", "Embarked_Q", "Embarked_S"]

X = df_encoded[feature_cols]
y = df_encoded["Survived"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

print(X_train.shape, y_train.shape)

torch.Size([712, 11]) torch.Size([712, 1])


## Defining the network

In [22]:
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(11, 4)
        self.output = nn.Linear(4, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

model = SimpleNet()
print(model)

SimpleNet(
  (hidden): Linear(in_features=11, out_features=4, bias=True)
  (output): Linear(in_features=4, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [23]:
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

epochs = 1000

for epoch in range(epochs):
    predictions = model(X_train)
    loss = criterion(predictions, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 200 == 0:
        print(f"epoch {epoch}, loss: {loss.item():.4f}")

epoch 0, loss: 0.6941
epoch 200, loss: 0.5922
epoch 400, loss: 0.4781
epoch 600, loss: 0.4570
epoch 800, loss: 0.4512


In [24]:
with torch.no_grad():
    test_predictions = model(X_test)
    test_predictions_binary = (test_predictions > 0.5).float()
    accuracy = (test_predictions_binary == y_test).float().mean()

print("accuracy:", round(accuracy.item(), 2))

accuracy: 0.81
